# Experiment 2 — Low-level image-statistic probes

A *deliberately engineered confound probe* in the sense of the review's Section
4.4: build predictors with **no access to lesion morphology** — only global
colour, exposure and geometry statistics of the frame — and measure how much
they recover.

Two questions:

1. **Is the acquisition centre decodable from image statistics?** If so the
   dataset carries a device/processor fingerprint (review §7.1) — the raw
   material for a "learned the site, not the disease" shortcut.
2. **Is the lesion class decodable from those same statistics?** A tree on colour
   histograms cannot find or characterise a subepithelial lesion. Any lift over
   chance is label information travelling a non-clinical route.

Two probe families (MLP and random forest) are run so the conclusion does not
hinge on one model class. All cross-validation is **grouped by patient**.

In [1]:
import sys, json
from collections import Counter
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path.cwd() / "experiments"))

import numpy as np
from PIL import Image

from cadsel.data import build_manifest
from probes import probe, fmt, grouped_folds, balanced_accuracy

OUT = Path("experiments/results")
OUT.mkdir(parents=True, exist_ok=True)
CACHE = OUT / "02_features.npz"

In [2]:
def features_for(path: str) -> list[float]:
    """Cheap global statistics. None of these encode lesion shape or position."""
    with Image.open(path) as im:
        im = im.convert("RGB")
        w, h = im.size
        small = np.asarray(im.resize((128, 128), Image.BILINEAR), dtype=np.float32) / 255.0

    f = [w, h, w / h, w * h]
    for i in range(3):
        ch = small[..., i]
        f += [ch.mean(), ch.std(),
              np.percentile(ch, 10), np.percentile(ch, 50), np.percentile(ch, 90)]
        hist, _ = np.histogram(ch, bins=8, range=(0, 1), density=True)
        f += list(hist)

    gray = small @ np.array([0.299, 0.587, 0.114], dtype=np.float32)
    f += [gray.mean(), gray.std()]
    f += [float((gray < 0.06).mean())]     # letterbox / field-of-view crop
    f += [float((gray > 0.94).mean())]     # specular highlight fraction
    mx, mn = small.max(axis=2), small.min(axis=2)
    f += [float((mx - mn).mean())]         # colourfulness
    lap = (gray[:-2, 1:-1] + gray[2:, 1:-1] + gray[1:-1, :-2] + gray[1:-1, 2:]
           - 4 * gray[1:-1, 1:-1])
    f += [float(lap.var())]                # sharpness / noise proxy
    return [float(v) for v in f]


NAMES = (["width", "height", "aspect", "pixels"]
         + [f"{c}_{s}" for c in "rgb"
            for s in ["mean", "std", "p10", "p50", "p90"] + [f"h{b}" for b in range(8)]]
         + ["gray_mean", "gray_std", "black_frac", "white_frac",
            "saturation", "laplacian_var"])

if CACHE.exists():
    z = np.load(CACHE, allow_pickle=True)
    X_all, meta_all = z["X"], z["meta"]
    print(f"loaded cached features {X_all.shape}")
else:
    feats, metas = [], []
    for modality in ("WLE", "EUS"):
        samples = build_manifest(modality)
        for i, s in enumerate(samples):
            feats.append(features_for(s.image))
            metas.append([modality, s.center, s.subtype,
                          "NET" if s.subtype.startswith("NET") else "non-NET",
                          s.patient_key])
            if (i + 1) % 750 == 0:
                print(f"  {modality} {i+1}/{len(samples)}")
    X_all = np.array(feats, dtype=np.float64)
    meta_all = np.array(metas, dtype=object)
    np.savez_compressed(CACHE, X=X_all, meta=meta_all)
    print(f"computed features {X_all.shape}")

print(f"{X_all.shape[1]} features per image, {len(NAMES)} names")
COLS = {n: i for i, n in enumerate(NAMES)}

  WLE 750/2817
  WLE 1500/2817
  WLE 2250/2817
  EUS 750/2095
  EUS 1500/2095
computed features (4912, 49)
49 features per image, 49 names


In [3]:
from sklearn.ensemble import RandomForestClassifier

def rf_probe(X, y_labels, groups, n_splits=5, seed=0):
    classes = sorted(set(map(str, y_labels)))
    idx = {c: i for i, c in enumerate(classes)}
    y = np.array([idx[str(v)] for v in y_labels])
    accs, bals, bases = [], [], []
    for tr, te in grouped_folds(groups, n_splits, seed):
        clf = RandomForestClassifier(n_estimators=300, min_samples_leaf=2,
                                     class_weight="balanced_subsample",
                                     random_state=seed, n_jobs=-1)
        clf.fit(X[tr], y[tr])
        pred = clf.predict(X[te])
        accs.append(float((pred == y[te]).mean()))
        bals.append(balanced_accuracy(y[te], pred, len(classes)))
        maj = np.bincount(y[tr], minlength=len(classes)).argmax()
        bases.append(float((y[te] == maj).mean()))
    return {
        "n_classes": len(classes),
        "accuracy": round(float(np.mean(accs)), 4),
        "balanced_accuracy": round(float(np.mean(bals)), 4),
        "majority_baseline": round(float(np.mean(bases)), 4),
        "chance_balanced": round(1 / len(classes), 4),
        "lift_over_majority": round(float(np.mean(accs) - np.mean(bases)), 4),
        "lift_over_chance_balanced": round(float(np.mean(bals) - 1 / len(classes)), 4),
    }


def subset(modality):
    m = meta_all[:, 0] == modality
    return X_all[m], meta_all[m]

## 2.1 Probe A — is the acquisition centre decodable?

In [4]:
results = []
for modality in ("WLE", "EUS"):
    X, M = subset(modality)
    groups = M[:, 4]
    for name, fn in (("MLP", probe), ("RF", rf_probe)):
        r = fn(X, M[:, 1], groups)
        r.update({"modality": modality, "probe": f"A/{name}", "target": "center",
                  "features": "all image stats"})
        results.append(r)
        print(fmt(f"{modality} {name}: centre from image stats", r))

WLE MLP: centre from image stats       acc 0.952 (base 0.808) | bal 0.754 (chance 0.167) | lift +0.587
WLE RF: centre from image stats        acc 0.974 (base 0.808) | bal 0.809 (chance 0.167) | lift +0.643
EUS MLP: centre from image stats       acc 0.942 (base 0.716) | bal 0.696 (chance 0.167) | lift +0.530
EUS RF: centre from image stats        acc 0.972 (base 0.716) | bal 0.782 (chance 0.167) | lift +0.615


## 2.2 Probe B — is the lesion class decodable from the same statistics?

The key test. These features cannot see a lesion.

In [5]:
for modality in ("WLE", "EUS"):
    X, M = subset(modality)
    groups = M[:, 4]
    for target, col in (("subtype", 2), ("is_net", 3)):
        for name, fn in (("MLP", probe), ("RF", rf_probe)):
            r = fn(X, M[:, col], groups)
            r.update({"modality": modality, "probe": f"B/{name}", "target": target,
                      "features": "all image stats"})
            results.append(r)
            print(fmt(f"{modality} {name}: {target} from image stats", r))

WLE MLP: subtype from image stats      acc 0.478 (base 0.621) | bal 0.248 (chance 0.200) | lift +0.049
WLE RF: subtype from image stats       acc 0.646 (base 0.621) | bal 0.254 (chance 0.200) | lift +0.054
WLE MLP: is_net from image stats       acc 0.642 (base 0.691) | bal 0.579 (chance 0.500) | lift +0.079
WLE RF: is_net from image stats        acc 0.728 (base 0.691) | bal 0.603 (chance 0.500) | lift +0.102
EUS MLP: subtype from image stats      acc 0.454 (base 0.693) | bal 0.228 (chance 0.200) | lift +0.028
EUS RF: subtype from image stats       acc 0.629 (base 0.693) | bal 0.198 (chance 0.200) | lift -0.002
EUS MLP: is_net from image stats       acc 0.630 (base 0.767) | bal 0.528 (chance 0.500) | lift +0.028
EUS RF: is_net from image stats        acc 0.723 (base 0.767) | bal 0.514 (chance 0.500) | lift +0.014


## 2.3 Probe C — colour/exposure only, geometry removed

Probe B mixes appearance with frame dimensions, and Experiment 1 already showed
geometry identifies the centre. Dropping the four geometry columns isolates
whether *appearance alone* carries label information.

In [6]:
APPEARANCE = [i for n, i in COLS.items()
              if n not in ("width", "height", "aspect", "pixels")]
for modality in ("WLE", "EUS"):
    X, M = subset(modality)
    Xa = X[:, APPEARANCE]
    groups = M[:, 4]
    for target, col in (("subtype", 2), ("is_net", 3), ("center", 1)):
        r = rf_probe(Xa, M[:, col], groups)
        r.update({"modality": modality, "probe": "C/RF", "target": target,
                  "features": "appearance only"})
        results.append(r)
        print(fmt(f"{modality} RF: {target} from appearance only", r))

WLE RF: subtype from appearance only   acc 0.643 (base 0.621) | bal 0.253 (chance 0.200) | lift +0.053
WLE RF: is_net from appearance only    acc 0.716 (base 0.691) | bal 0.588 (chance 0.500) | lift +0.087
WLE RF: center from appearance only    acc 0.956 (base 0.808) | bal 0.642 (chance 0.167) | lift +0.476
EUS RF: subtype from appearance only   acc 0.608 (base 0.693) | bal 0.199 (chance 0.200) | lift -0.001
EUS RF: is_net from appearance only    acc 0.711 (base 0.767) | bal 0.496 (chance 0.500) | lift -0.004
EUS RF: center from appearance only    acc 0.733 (base 0.716) | bal 0.447 (chance 0.167) | lift +0.280


## 2.4 Which statistics carry the signal?

In [7]:
for modality in ("WLE", "EUS"):
    X, M = subset(modality)
    for target, col in (("center", 1), ("subtype", 2)):
        clf = RandomForestClassifier(n_estimators=300, min_samples_leaf=2,
                                     class_weight="balanced_subsample",
                                     random_state=0, n_jobs=-1)
        clf.fit(X, M[:, col])
        order = np.argsort(clf.feature_importances_)[::-1][:8]
        top = ", ".join(f"{NAMES[i]} {100*clf.feature_importances_[i]:.1f}%"
                        for i in order)
        print(f"{modality} -> {target:<8}: {top}")

WLE -> center  : width 11.1%, pixels 10.2%, height 9.1%, aspect 6.3%, r_h0 5.5%, r_p10 5.5%, g_p10 5.0%, b_p10 4.1%
WLE -> subtype : black_frac 3.3%, laplacian_var 3.3%, r_h7 2.9%, b_mean 2.8%, r_h0 2.8%, b_h3 2.6%, saturation 2.5%, b_std 2.4%
EUS -> center  : height 12.1%, aspect 10.3%, width 10.2%, pixels 9.5%, saturation 5.2%, laplacian_var 3.9%, r_std 3.2%, r_h3 3.1%
EUS -> subtype : laplacian_var 6.8%, b_std 3.5%, r_std 3.3%, g_std 3.1%, gray_std 3.0%, black_frac 2.8%, r_h3 2.8%, g_h3 2.7%


## 2.5 Verdict

In [8]:
json.dump(results, open(OUT / "02_lowlevel_probes.json", "w"), indent=2)

print("=" * 84)
print("LOW-LEVEL PROBES — balanced accuracy vs uniform chance")
print("=" * 84)
hdr = f"{'modality':<9}{'probe':<9}{'features':<18}{'target':<9}{'bal':>7}{'chance':>8}{'lift':>8}"
print(hdr)
print("-" * len(hdr))
for r in results:
    flag = "  <-- decodable" if r["lift_over_chance_balanced"] > 0.10 else ""
    print(f"{r['modality']:<9}{r['probe']:<9}{r['features']:<18}{r['target']:<9}"
          f"{r['balanced_accuracy']:>7.3f}{r['chance_balanced']:>8.3f}"
          f"{r['lift_over_chance_balanced']:>+8.3f}{flag}")

centre = [r for r in results if r["target"] == "center"]
klass = [r for r in results if r["target"] in ("subtype", "is_net")]
print(f"\ncentre probes : max lift {max(r['lift_over_chance_balanced'] for r in centre):+.3f}")
print(f"class probes  : max lift {max(r['lift_over_chance_balanced'] for r in klass):+.3f}")

LOW-LEVEL PROBES — balanced accuracy vs uniform chance
modality probe    features          target       bal  chance    lift
--------------------------------------------------------------------
WLE      A/MLP    all image stats   center     0.754   0.167  +0.587  <-- decodable
WLE      A/RF     all image stats   center     0.809   0.167  +0.643  <-- decodable
EUS      A/MLP    all image stats   center     0.696   0.167  +0.530  <-- decodable
EUS      A/RF     all image stats   center     0.782   0.167  +0.615  <-- decodable
WLE      B/MLP    all image stats   subtype    0.248   0.200  +0.049
WLE      B/RF     all image stats   subtype    0.254   0.200  +0.054
WLE      B/MLP    all image stats   is_net     0.579   0.500  +0.079
WLE      B/RF     all image stats   is_net     0.603   0.500  +0.102  <-- decodable
EUS      B/MLP    all image stats   subtype    0.228   0.200  +0.028
EUS      B/RF     all image stats   subtype    0.198   0.200  -0.002
EUS      B/MLP    all image stats   is_net